# Auto Re-ID — zero typing

Cleans tracker IDs **fully automatically**. No gallery, no Excel, no manual labelling.

**How it works (learned from your data):**
- Most tracker IDs are already clean — each follows one player for the whole half (100% class & team purity). They're kept as-is.
- A few "junk" IDs flip between players and the referee constantly (class purity < 85%). Their boxes are auto-resolved:
  - boxes that **duplicate** a clean player (same team, close by) → dropped
  - **referee** boxes → merged into one referee ID
  - **unique** boxes → re-stitched onto a clean player ID that went missing nearby, else dropped

**Output:** `per_frame_tracks_clean.csv` + an annotated video (GPU-encoded) so you can verify the result.

Run top to bottom. Enable **T4 GPU** (`Runtime → Change runtime type`) for fast video encoding.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
VIDEO_PATH   = "/content/HIL-HAZ_half1.mp4"                       # input video
TRACKS_CSV   = "/content/per_frame_tracks_HILHAZ_half1.csv"       # tracker CSV
CLEAN_CSV    = "/content/per_frame_tracks_clean.csv"             # output CSV
ANNO_OUT     = "/content/annotated_clean.mp4"                    # output video

# Track classification
CLASS_PURITY_MIN = 0.85   # track is 'clean' if dominant class ≥ this; else 'junk'

# Junk-box resolution
DUP_PX     = 90    # junk box within this px of a same-team clean box ⇒ duplicate → drop
STITCH_PX  = 220   # re-stitch a unique junk box onto a clean ID missing within this px
STITCH_GAP = 45    # …if that clean ID disappeared no more than this many frames ago
REF_ID     = 99    # canonical ID assigned to all referee detections

# Gap interpolation
MAX_GAP = 25       # fill detection gaps up to this many frames per ID

# Video render window
FRAME_START  = 0
FRAME_END    = None    # None = whole video
RENDER_EVERY = 1
VIDEO_BITRATE = "8M"

BALL_CLASS, GK_CLASS, PLAYER_CLASS, REF_CLASS = 0, 1, 2, 3
PEOPLE = (GK_CLASS, PLAYER_CLASS, REF_CLASS)

In [ ]:
# ── Load + dedupe + classify tracks ───────────────────────────────────────────
import subprocess, sys
for pkg in ["pandas", "numpy"]:
    try: __import__(pkg)
    except ImportError: subprocess.run([sys.executable,"-m","pip","install","-q",pkg])
import pandas as pd, numpy as np

df = pd.read_csv(TRACKS_CSV, encoding_errors="replace", low_memory=False)

# Dedupe (frame, tracker_id) — keep highest-confidence box
_b = len(df)
if "conf" in df.columns:
    df = df.sort_values("conf", ascending=False)
df = (df.drop_duplicates(["frame","display_track_id"], keep="first")
        .sort_values(["frame","display_track_id"]).reset_index(drop=True))
print(f"Loaded {_b:,} rows → {len(df):,} after dedupe ({_b-len(df):,} dropped)")

df["cx"] = (df["x1"] + df["x2"]) / 2
df["cy"] = (df["y1"] + df["y2"]) / 2

ppl = df[df["class_id"].isin(PEOPLE)]

# Classify each tracker ID: clean (one consistent person) vs junk (catch-all)
clean_ids, junk_ids, track_meta = [], [], {}
for tid, g in ppl.groupby("display_track_id"):
    cls_pure  = g["class_id"].value_counts(normalize=True).iloc[0]
    mode_cls  = int(g["class_id"].mode().iloc[0])
    mode_team = int(g["team_id"].mode().iloc[0])
    track_meta[int(tid)] = {"cls": mode_cls, "team": mode_team, "purity": cls_pure}
    (clean_ids if cls_pure >= CLASS_PURITY_MIN else junk_ids).append(int(tid))

print(f"\nClean tracker IDs ({len(clean_ids)}): {sorted(clean_ids)}")
print(f"Junk  tracker IDs ({len(junk_ids)}): {sorted(junk_ids)}")
for j in sorted(junk_ids):
    m = track_meta[j]
    print(f"   junk {j}: class purity {m['purity']:.0%} → will auto-resolve its boxes")

In [ ]:
# ── Auto-resolve junk boxes ────────────────────────────────────────────────────
# Clean tracks keep their ID. Each junk box is either dropped (duplicate),
# tagged referee, re-stitched onto a recently-missing clean ID, or dropped.
try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **k): return x

df["clean_id"] = df["display_track_id"]
clean_set = set(clean_ids)
junk_set  = set(junk_ids)

# last-seen position of each clean ID (updated as we sweep frames forward)
last_seen = {}   # clean_id → (frame, cx, cy, team)
drop_idx  = []
stats = {"dup_drop": 0, "ref": 0, "stitch": 0, "unique_drop": 0}

frames = sorted(ppl["frame"].unique())
ppl_by_frame = {f: g for f, g in df[df["class_id"].isin(PEOPLE)].groupby("frame")}

for f in tqdm(frames, desc="resolve", unit="f"):
    g = ppl_by_frame[f]
    clean_here = g[g["display_track_id"].isin(clean_set)]

    # update last-seen for clean IDs present this frame
    for _, r in clean_here.iterrows():
        last_seen[int(r.display_track_id)] = (f, float(r.cx), float(r.cy), int(r.team_id))
    present_clean = set(clean_here["display_track_id"].astype(int))

    # arrays for fast duplicate test
    cxs = clean_here["cx"].values; cys = clean_here["cy"].values
    cts = clean_here["team_id"].values

    junk_here = g[g["display_track_id"].isin(junk_set)].sort_values("conf", ascending=False)
    used_this_frame = set()
    for idx, r in junk_here.iterrows():
        cls = int(r.class_id); team = int(r.team_id)
        jx, jy = float(r.cx), float(r.cy)

        # referee → single referee ID
        if cls == REF_CLASS:
            df.at[idx, "clean_id"] = REF_ID
            stats["ref"] += 1
            continue

        # duplicate of a clean same-team box present now → drop
        if len(cxs):
            d = np.hypot(cxs - jx, cys - jy)
            same = (cts == team)
            if np.any(same) and np.min(d[same]) < DUP_PX:
                drop_idx.append(idx); stats["dup_drop"] += 1; continue

        # re-stitch onto a clean ID of same team that vanished recently near here
        best, best_d = None, STITCH_PX
        for cid, (lf, lx, ly, lt) in last_seen.items():
            if cid in present_clean or cid in used_this_frame:  continue
            if lt != team:                                      continue
            if f - lf > STITCH_GAP:                              continue
            dd = float(np.hypot(lx - jx, ly - jy))
            if dd < best_d:
                best, best_d = cid, dd
        if best is not None:
            df.at[idx, "clean_id"] = best
            last_seen[best] = (f, jx, jy, team)
            used_this_frame.add(best)
            stats["stitch"] += 1
        else:
            drop_idx.append(idx); stats["unique_drop"] += 1

df = df.drop(index=drop_idx)
df["display_track_id"] = df["clean_id"]
df = df.drop(columns="clean_id")

print("\nJunk-box resolution:")
print(f"   duplicates dropped : {stats['dup_drop']:,}")
print(f"   referee boxes      : {stats['ref']:,}  → ID {REF_ID}")
print(f"   re-stitched        : {stats['stitch']:,}  → onto clean player IDs")
print(f"   unique dropped     : {stats['unique_drop']:,}")
print(f"\nCanonical IDs now: {sorted(df[df['class_id'].isin(PEOPLE)]['display_track_id'].unique())}")

In [ ]:
# ── Resolve collisions + interpolate gaps + save ──────────────────────────────
# Same canonical ID twice in one frame → keep highest-confidence box
ppl_mask = df["class_id"].isin(PEOPLE)
coll = (df[ppl_mask].groupby(["frame","display_track_id"]).size()
        .reset_index(name="k").query("k > 1"))
if len(coll):
    df["_p"] = ppl_mask
    keep = (df[df["_p"]].sort_values("conf", ascending=False)
            .drop_duplicates(["frame","display_track_id"], keep="first").index)
    df = pd.concat([df[~df["_p"]], df.loc[keep]]).sort_index().drop(columns="_p")
print(f"Frame collisions resolved: {len(coll):,}")

# Interpolate short gaps per canonical ID
def _interp(d, max_gap=MAX_GAP):
    d = d.copy()
    if "track_filled" not in d.columns: d["track_filled"] = 0
    lin = [c for c in ["x1","y1","x2","y2","x_m","y_m","cx","cy"] if c in d.columns]
    new = []
    for cid, g in d[d["class_id"].isin(PEOPLE)].groupby("display_track_id"):
        g = g.sort_values("frame"); fr = g["frame"].values
        for a, b in zip(fr[:-1], fr[1:]):
            gap = int(b - a)
            if 1 < gap <= max_gap:
                ra = g[g["frame"]==a].iloc[0]; rb = g[g["frame"]==b].iloc[0]
                for k in range(1, gap):
                    t = k/gap; row = ra.copy(); row["frame"] = a+k
                    for c in lin:
                        if pd.notna(ra[c]) and pd.notna(rb[c]):
                            row[c] = ra[c] + t*(rb[c]-ra[c])
                    row["conf"] = 0.0; row["track_filled"] = 1
                    if "detector_ran" in row.index: row["detector_ran"] = 0
                    new.append(row)
    if new: d = pd.concat([d, pd.DataFrame(new)], ignore_index=True)
    return d.sort_values(["frame","display_track_id"]).reset_index(drop=True)

df = _interp(df)
df.to_csv(CLEAN_CSV, index=False)
ppl_out = df[df["class_id"].isin(PEOPLE)]
print(f"\n✅ Saved → {CLEAN_CSV}  ({len(df):,} rows, "
      f"{int(df.get('track_filled', pd.Series([0])).sum()):,} interpolated)")
print(f"Consistent IDs: {ppl_out['display_track_id'].nunique()}")
print("\nFrames covered per ID:")
print(ppl_out.groupby(["display_track_id","team_id"])["frame"].nunique()
      .sort_values(ascending=False).to_string())

In [ ]:
# ── Render annotated video (GPU / NVENC) ──────────────────────────────────────
import cv2, os, subprocess
try: from tqdm.auto import tqdm
except Exception:
    def tqdm(x,**k): return x

# Check NVENC; fall back to CPU mp4v if unavailable
_enc = subprocess.run(["ffmpeg","-hide_banner","-encoders"],
                      capture_output=True, text=True).stdout
USE_NVENC = "h264_nvenc" in _enc
print("Encoder:", "h264_nvenc (GPU)" if USE_NVENC else "mp4v (CPU — enable T4 GPU for speed)")

_COL = {
    (PLAYER_CLASS, 0):(240,100,180),(PLAYER_CLASS, 1):(30,210,240),(PLAYER_CLASS,-1):(160,160,160),
    (GK_CLASS, 0):(180,60,255),(GK_CLASS, 1):(80,200,120),(GK_CLASS,-1):(200,200,60),
    (REF_CLASS,-1):(255,140,0),(REF_CLASS,0):(255,140,0),(REF_CLASS,1):(255,140,0),
}
_CLS_LBL = {PLAYER_CLASS:"P", GK_CLASS:"GK", REF_CLASS:"REF"}
BALL_COL = (255,220,0)
def _col(c,t): return _COL.get((int(c),int(t)), _COL.get((int(c),-1),(160,160,160)))
def _lab(fr,txt,ox,oy,sc,th,col):
    (tw,h),bl = cv2.getTextSize(txt,cv2.FONT_HERSHEY_SIMPLEX,sc,th)
    cv2.rectangle(fr,(ox-2,oy-h-bl-2),(ox+tw+2,oy+2),(0,0,0),-1)
    cv2.putText(fr,txt,(ox,oy),cv2.FONT_HERSHEY_SIMPLEX,sc,col,th,cv2.LINE_AA)

grp  = df.set_index("frame")
cap  = cv2.VideoCapture(VIDEO_PATH)
fps  = cap.get(cv2.CAP_PROP_FPS) or 25.0
tot  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W    = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
f0   = int(FRAME_START); f1 = min((int(FRAME_END) if FRAME_END is not None else tot-1), tot-1)
ofps = fps / max(1, RENDER_EVERY); sc = W/1920
os.makedirs(os.path.dirname(ANNO_OUT) or ".", exist_ok=True)

if USE_NVENC:
    proc = subprocess.Popen(
        ["ffmpeg","-y","-f","rawvideo","-vcodec","rawvideo","-s",f"{W}x{H}",
         "-pix_fmt","bgr24","-r",str(ofps),"-i","pipe:0","-c:v","h264_nvenc",
         "-preset","fast","-b:v",VIDEO_BITRATE,"-pix_fmt","yuv420p",ANNO_OUT],
        stdin=subprocess.PIPE)
    writer = None
else:
    writer = cv2.VideoWriter(ANNO_OUT, cv2.VideoWriter_fourcc(*"mp4v"), ofps, (W,H))
    proc = None

cap.set(cv2.CAP_PROP_POS_FRAMES, f0)
n = 0
for fi in tqdm(range(f0, f1+1), desc="render", unit="f"):
    ok, frame = cap.read()
    if not ok: break
    if (fi-f0) % RENDER_EVERY != 0: continue
    cv2.rectangle(frame,(0,0),(360,42),(0,0,0),-1)
    cv2.putText(frame,f"{fi:05d}   {fi/fps:.2f}s",(8,30),
                cv2.FONT_HERSHEY_SIMPLEX,1.0,(255,220,0),2,cv2.LINE_AA)
    if fi in grp.index:
        rows = grp.loc[[fi]]
        for _, br in rows[rows["class_id"]==BALL_CLASS].iterrows():
            cv2.circle(frame,(int((br.x1+br.x2)/2),int((br.y1+br.y2)/2)),
                       max(6,int(8*sc)),BALL_COL,2,cv2.LINE_AA)
        for _, r in rows[rows["class_id"].isin(PEOPLE)].iterrows():
            tid=int(r.display_track_id); cls=int(r.class_id)
            team=int(r.team_id) if pd.notna(r.team_id) else -1
            col=_col(cls,team); x1,y1,x2,y2=int(r.x1),int(r.y1),int(r.x2),int(r.y2)
            cv2.rectangle(frame,(x1,y1),(x2,y2),col,max(1,int(2*sc+1)))
            tm = "" if team<0 else f"t{team}"
            _lab(frame,str(tid),x1,max(0,y1-20),0.85*max(sc,0.6),2,col)
            _lab(frame,f"{_CLS_LBL.get(cls,'?')}{tm}",x1,max(0,y1-2),0.5*max(sc,0.5),1,col)
    if proc: proc.stdin.write(frame.tobytes())
    else:    writer.write(frame)
    n += 1

cap.release()
if proc: proc.stdin.close(); proc.wait()
else:    writer.release()
print(f"\n✅ {n} frames → {ANNO_OUT}  ({os.path.getsize(ANNO_OUT)/1e6:.1f} MB)")

In [ ]:
# ── Download results ──────────────────────────────────────────────────────────
from google.colab import files as colab_files
colab_files.download(CLEAN_CSV)
colab_files.download(ANNO_OUT)